# GSS missing values

The 2016-2024 job-satisfaction subset mixes two kinds of empty cells:

- **Inapplicable.** The question was skipped because of work status or ballot. These are not answers.
- **Item nonresponse.** The question was asked and the person did not answer (don't know or refused). Income and health are the main cases.

Replacing either kind with the string `"N/A"` would turn numeric columns into text and would treat a skipped question as if it were a real response. Means and medians are also not filled here. Those fills have to be fit on the training fold only, inside the modeling pipeline.

What this notebook does: show where the holes are, then call `clean_missing.py` to keep current workers with a valid `SATJOB` and an occupation code. Leftover nonresponse stays as NaN.

## Imports and loading data

In [1]:
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

cwd = Path.cwd()
subset_rel = Path("data") / "interim" / "gss_job_satisfaction_subset.parquet"
if (cwd / subset_rel).exists():
    PROJECT_ROOT = cwd
elif (cwd.parent.parent / subset_rel).exists():
    PROJECT_ROOT = cwd.parent.parent
else:
    raise FileNotFoundError(
        "Could not find data/interim/gss_job_satisfaction_subset.parquet. "
        "Run this notebook from the project root."
    )

sys.path.insert(0, str(PROJECT_ROOT / "notebooks" / "gss_eda"))
from clean_missing import (
    INPUT_PARQUET,
    OUTPUT_PARQUET,
    clean_gss,
    describe_row_drops,
    null_summary,
    write_clean,
)

df = pd.read_parquet(INPUT_PARQUET)
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns.")

Loaded 16,100 rows and 20 columns.


## Overall missingness

`HRS2`, `JOBLOSE`, `HRS1`, income, and `SATJOB` look mostly empty in the full subset. The tables below show that those rates are skip patterns, not random holes spread across every respondent.

In [2]:
overall_missing = pd.DataFrame(
    {
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(2),
    }
).sort_values("pct_missing", ascending=False)
overall_missing

,n_missing,pct_missing
HRS2,15769,97.94
JOBLOSE,9920,61.61
HRS1,7251,45.04
REALRINC,6783,42.13
RINCOME,6783,42.13
SATJOB,5141,31.93
REALINC,1802,11.19
HEALTH,1788,11.11
OCC10,791,4.91
AGE,659,4.09


## Job satisfaction by work status

`SATJOB` is not asked of people who are retired, in school, or coded "other" (`WRKSTAT` 5, 6, and 8). Full-time, part-time, and temporarily not at work are only a few percent missing. Unemployed respondents and people keeping house were usually asked, but that answer is not about a current job we can match to O*NET, so they stay out of this modeling file.

In [3]:
wrkstat_labels = {
    1: "1 Full-time",
    2: "2 Part-time",
    3: "3 Job, not at work",
    4: "4 Unemployed",
    5: "5 Retired",
    6: "6 In school",
    7: "7 Keeping house",
    8: "8 Other",
}
satjob_by_work = pd.crosstab(
    df["WRKSTAT"].map(wrkstat_labels).fillna("Missing work status"),
    df["SATJOB"].isna().map({False: "SATJOB answered", True: "SATJOB missing"}),
    dropna=False,
)
satjob_by_work

SATJOB,SATJOB answered,SATJOB missing
WRKSTAT,,
1 Full-time,7040,250
2 Part-time,1565,70
"3 Job, not at work",328,37
4 Unemployed,736,83
5 Retired,0,3581
6 In school,0,436
7 Keeping house,1286,125
8 Other,0,532
Missing work status,4,27


## Job-loss question by ballot

`JOBLOSE` was never asked on ballot 3. Among employed people on ballots 1 and 2 it is nearly complete, but using it in the main file would mean dropping a random third of the sample or inventing a job-loss answer for people who were not asked. It is left out of the cleaned file.

In [4]:
joblose_by_ballot = (
    df.groupby("BALLOT")["JOBLOSE"]
    .apply(
        lambda s: pd.Series(
            {"n": len(s), "pct_missing": round(s.isna().mean() * 100, 2)}
        )
    )
    .unstack()
)
display(joblose_by_ballot)

joblose_by_year_ballot = (
    df.groupby(["YEAR", "BALLOT"])["JOBLOSE"]
    .apply(lambda s: round(s.isna().mean() * 100, 1))
    .unstack()
)
joblose_by_year_ballot

,n,pct_missing
BALLOT,,
1.0,5345.0,41.87
2.0,5378.0,42.86
3.0,5377.0,100.00


BALLOT,1.0,2.0,3.0
YEAR,,,
2016,41.5,40.3,100.0
2018,40.3,38.1,100.0
2021,44.6,45.4,100.0
2022,39.5,43.4,100.0
2024,42.6,44.8,100.0


## Hours

`HRS2` is asked only when someone has a job but did not work last week (`WRKSTAT` 3). `HRS1` covers full-time and part-time workers. The cleaning step folds them into one `HOURS` column.

In [5]:
print(
    "HRS1 missing among full-time and part-time:",
    f"{df.loc[df['WRKSTAT'].isin([1, 2]), 'HRS1'].isna().mean() * 100:.1f}%",
)
print("Work status of people with a non-missing HRS2:")
print(df.loc[df["HRS2"].notna(), "WRKSTAT"].value_counts(dropna=False).sort_index())

HRS1 missing among full-time and part-time: 0.9%
Work status of people with a non-missing HRS2:
WRKSTAT
3.0    331
Name: count, dtype: int64


## Missingness among people currently working

After limiting to `WRKSTAT` 1, 2, or 3, the large skip-pattern holes shrink. What remains on income, health, age, and a few demographics is real nonresponse. Those cells stay missing. Dropping every row with a missing income or health value would cut the sample and would drop people who refuse those questions, who are not a random subset.

In [6]:
employed = df[df["WRKSTAT"].isin([1, 2, 3])]
print(f"Currently working: {len(employed):,} rows")
employed_missing = pd.DataFrame(
    {
        "n_missing": employed.isna().sum(),
        "pct_missing": (employed.isna().mean() * 100).round(2),
    }
).sort_values("pct_missing", ascending=False)
employed_missing

Currently working: 9,290 rows


,n_missing,pct_missing
HRS2,8959,96.44
JOBLOSE,3110,33.48
RINCOME,1488,16.02
REALRINC,1488,16.02
HEALTH,1095,11.79
REALINC,774,8.33
HRS1,441,4.75
AGE,374,4.03
SATJOB,357,3.84
OCC10,154,1.66


## Cleaning rules

Applied by `clean_missing.clean_gss` (this notebook does not reimplement them):

- Keep rows with `WRKSTAT` in {1, 2, 3}, `SATJOB` in {1, 2, 3, 4}, and a non-missing `OCC10`.
- Do not impute `SATJOB`. Rows without a satisfaction answer cannot be used for supervised learning.
- Set `HOURS` from `HRS1` for full-time and part-time workers, and from `HRS2` when `WRKSTAT` is 3. Then drop `HRS1` and `HRS2`.
- Drop `JOBLOSE` and `RINCOME`. `RINCOME` is the income bracket that `REALRINC` is built from, with the same missingness.
- Keep `WTSSNRPS` for weighted descriptives. It is not a predictor.
- Leave remaining NaNs on age, sex, race, marital status, health, happiness, self-employment, both income variables, and hours. Impute those later inside the training fold (and add a missing indicator for income and health). Tree models can keep NaN; logistic regression, k-nearest neighbors, and the neural net cannot.

In [7]:
print(describe_row_drops(df).to_string(index=False))

cleaned = clean_gss(df)
write_clean(cleaned, OUTPUT_PARQUET)
print(f"\nCleaned shape: {cleaned.shape[0]:,} rows, {cleaned.shape[1]} columns")
print(f"Wrote {OUTPUT_PARQUET}")
null_summary(cleaned)

                                  Step  Rows
                         Starting rows 16100
Currently working (WRKSTAT 1, 2, or 3)  9290
             ...and valid SATJOB (1-4)  8933
       ...and non-missing OCC10 (kept)  8783

Cleaned shape: 8,783 rows, 17 columns
Wrote C:\Users\jessi\BTT\Swytch-2B-predicting-job-satisfaction-from-person-job-fit\data\processed\gss_job_satisfaction_clean.parquet


,n_missing,pct_missing
REALRINC,1304,14.85
HEALTH,1076,12.25
REALINC,690,7.86
AGE,320,3.64
RACE,83,0.95
SEX,62,0.71
HAPPY,35,0.40
HOURS,23,0.26
MARITAL,16,0.18
WRKSLF,8,0.09
